In [1]:
import numpy as np
import pywt
from scipy.stats import skew, kurtosis
import pandas as pd


In [ ]:
def extraer_features(latidos, picos_R, fs=500):
    features = []

    # ── 1. TEMPORALES / HRV ───────────────────────────────────────────────────
    intervalos_RR = np.diff(picos_R) / fs
    features.append(np.mean(intervalos_RR))
    features.append(np.std(intervalos_RR))
    features.append(np.min(intervalos_RR))
    features.append(np.max(intervalos_RR))
    features.append(np.max(intervalos_RR) - np.min(intervalos_RR))
    features.append(60 / np.mean(intervalos_RR))
    rmssd = np.sqrt(np.mean(np.diff(intervalos_RR) ** 2)) if len(intervalos_RR) > 1 else 0.0
    features.append(rmssd)

    # ── 2. MORFOLÓGICAS ───────────────────────────────────────────────────────
    amp_R, amp_T, area_QRS, dur_QRS, amp_Q, st_nivel = [], [], [], [], [], []
    for latido in latidos:
        pico_R_idx = 100
        amp_R.append(latido[pico_R_idx])
        amp_T.append(np.max(latido[150:250]))
        zona_QRS = latido[75:125]
        area_QRS.append(np.trapz(np.abs(zona_QRS)))
        umbral_QRS = 0.5 * latido[pico_R_idx]
        dur_QRS.append(np.sum(zona_QRS > umbral_QRS))
        amp_Q.append(np.min(latido[75:100]))
        st_nivel.append(np.mean(latido[125:155]))

    for stat_list in [amp_R, amp_T, area_QRS, dur_QRS, amp_Q, st_nivel]:
        features.append(np.mean(stat_list))
        features.append(np.std(stat_list))

    # ── 3. ESTADÍSTICAS ───────────────────────────────────────────────────────
    for fn in [np.mean, np.std, skew, kurtosis, np.max, np.min]:
        vals = [fn(l) for l in latidos]
        features.append(np.mean(vals))
        features.append(np.std(vals))

    # ── 4. FRECUENCIALES ──────────────────────────────────────────────────────
    energias_d4, energias_d5, energias_d6 = [], [], []
    for latido in latidos:
        coeffs = pywt.wavedec(latido, 'db6', level=8)
        energias_d4.append(np.sum(coeffs[5] ** 2))
        energias_d5.append(np.sum(coeffs[4] ** 2))
        energias_d6.append(np.sum(coeffs[3] ** 2))

    for energias in [energias_d4, energias_d5, energias_d6]:
        features.append(np.mean(energias))
        features.append(np.std(energias))

    ratio_LF_HF = np.mean(energias_d6) / (np.mean(energias_d4) + 1e-8)
    features.append(ratio_LF_HF)

    return np.array(features)

In [ ]:
Y = np.load('Y.npy')
picos_R_pacientes = np.load('picos_R.npy', allow_pickle=True)
latidos_pacientes = np.load('latidos.npy', allow_pickle=True)
latidos_12_pacientes = np.load('latidos_12.npy', allow_pickle=True) #El preprocesado de los latidos peta al intentar guardar este array, por lo que vamos a intentar es pasarlo el codigo de extraccion de caracterisitcas al fichero de preprocesado de los latidos, y asi solo guardar el array de caracteristicas, que es mucho mas pequeño que el de los latidos.
pacientes_validos = np.load('pacientes_validos.npy', allow_pickle=True)

df_pacientes = pd.read_csv("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\ptbxl_database.csv")

print(f"Pacientes: {len(pacientes_validos)}")

In [ ]:
"""
X_feat = []

for i, record in enumerate(pacientes_validos):
    latidos  = latidos_pacientes[i].astype(np.float64)
    picos_R  = picos_R_pacientes[i]
    features = extraer_features(latidos, picos_R, fs=500)

    fila = df_pacientes.loc[
        df_pacientes['filename_hr'].str.contains(record, case=False, na=False)
    ]
    edad = fila['age'].values[0]
    sexo = 1.0 if fila['sex'].values[0] == 1 else 0.0
    vector_completo = np.append(features, [edad, sexo])
    X_feat.append(vector_completo)

X_feat = np.array(X_feat)

print("Shape X_feat:", X_feat.shape)  # debería ser (N, 40)
print("Shape Y:     ", Y.shape)

np.save('X_feat.npy', X_feat)
print("Guardado correctamente")
"""

C:\Users\Ana\AppData\Local\Temp\ipykernel_19024\831460112.py:35: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area_QRS.append(np.trapz(np.abs(zona_QRS)))
c:\Users\Ana\AppData\Local\Programs\Python\Python312\Lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 8 is too high: all coefficients will experience boundary effects.
  warnings.warn(


Shape X_feat: (16984, 40)
Shape Y:      (16984, 5)
Guardado correctamente


In [ ]:
X_feat    = []  # features solo derivación I → (N, 40)
X_feat_12 = []  # features 12 derivaciones  → (N, 12×38+2 = 458)

for i, record in enumerate(pacientes_validos):
    fila = df_pacientes.loc[
        df_pacientes['filename_hr'].str.contains(record, case=False, na=False)
    ]
    if len(fila) == 0:
        continue

    picos_R    = picos_R_pacientes[i]
    edad = fila['age'].values[0]
    sexo = 1.0 if fila['sex'].values[0] == 1 else 0.0

    # ── Features derivación I ─────────────────────────────────────────────────
    latidos_I  = latidos_pacientes[i].astype(np.float64)
    features_I = extraer_features(latidos_I, picos_R, fs=500)
    X_feat.append(np.append(features_I, [edad, sexo]))

    # ── Features 12 derivaciones ──────────────────────────────────────────────
    # Para cada derivación extraemos las mismas features y las concatenamos
    # Las features temporales/HRV son iguales en todas las derivaciones
    # (dependen de los picos R, no de la morfología)
    # así que las calculamos solo una vez y las añadimos al principio
    latidos_12 = latidos_12_pacientes[i]  # shape (12, MAX_LATIDOS, LONGITUD_LATIDO)

    features_todas = []
    for d in range(12):
        latidos_d  = latidos_12[d].astype(np.float64)
        features_d = extraer_features(latidos_d, picos_R, fs=500)
        features_todas.append(features_d)

    features_concat = np.concatenate(features_todas)  # 12 × 38 = 456 features
    X_feat_12.append(np.append(features_concat, [edad, sexo]))

X_feat    = np.array(X_feat)
X_feat_12 = np.array(X_feat_12)

print("Shape X_feat: ", X_feat.shape)    # (N, 40)
print("Shape X_feat_12:", X_feat_12.shape) # (N, 458)
print("Shape Y: ", Y.shape)

np.save('X_feat.npy', X_feat)
np.save('X_feat_12.npy', X_feat_12)
print("Guardado correctamente")